In [ ]:
import sys, os
os.environ['OMP_NUM_THREADS'] = '1'
os.environ['OPENBLAS_NUM_THREADS'] = '1'
os.environ['MKL_NUM_THREADS'] = '1'
os.environ['VECLIB_MAXIMUM_THREADS'] = '1'

import utils as ut
import meshio
import numpy as np
import pandas as pd
from pathlib import Path

In [ ]:
datadir = "./"
inname = "Nicoya_interface.e"
outname = "nicoya_rotated_corrected.geo"
geofile = datadir + outname

In [ ]:
mesh = meshio.read(datadir + inname)
points = mesh.points
print(mesh.cells)

In [ ]:
# ROTATION: Apply counterclockwise 45-degree rotation to points
print("Original points shape:", points.shape)
print("Original point range (before rotation):")
print(f"x: [{points[:,0].min():.3f}, {points[:,0].max():.3f}]")
print(f"y: [{points[:,1].min():.3f}, {points[:,1].max():.3f}]")
print(f"z: [{points[:,2].min():.3f}, {points[:,2].max():.3f}]")

# Define rotation matrix for 45 degrees counterclockwise
angle_deg = 45
angle_rad = np.radians(angle_deg)
cos_a = np.cos(angle_rad)
sin_a = np.sin(angle_rad)

rotation_matrix = np.array([
    [cos_a, -sin_a],
    [sin_a,  cos_a]
])

print(f"\nApplying {angle_deg}° counterclockwise rotation...")
print("Rotation matrix:")
print(rotation_matrix)

# Apply rotation to x,y coordinates (keeping z unchanged)
xy_original = points[:, :2].T
xy_rotated = rotation_matrix @ xy_original
points_rotated = np.column_stack([xy_rotated.T, points[:, 2]])

print("\nRotated point range:")
print(f"x: [{points_rotated[:,0].min():.3f}, {points_rotated[:,0].max():.3f}]")
print(f"y: [{points_rotated[:,1].min():.3f}, {points_rotated[:,1].max():.3f}]")
print(f"z: [{points_rotated[:,2].min():.3f}, {points_rotated[:,2].max():.3f}]")

# Use rotated points for the rest of the process
points = points_rotated

In [ ]:
# Create DataFrame with rotated points
df = pd.DataFrame(points, columns=["x", "y", "z"])

# Print ranges
ranges = df.agg(["min", "max"])
print("DataFrame ranges (rotated coordinates):")
print(ranges)

In [ ]:
# Define domain bounds - KEEP ORIGINAL, don't adjust for rotation
sep = "//+"
xmin, xmax = -1000e3, 1000e3
ymin, ymax = -1000e3, 1000e3
zmin, zmax = -400e3, 0.0

print(f"Domain bounds (original): x=[{xmin}, {xmax}], y=[{ymin}, {ymax}], z=[{zmin}, {zmax}]")

# Calculate offset based on rotated coordinates
x0 = (df['x'].min() + df['x'].max()) / 2
y0 = (df['y'].min() + df['y'].max()) / 2
print(f"Calculated offset: x0={x0:.1f}, y0={y0:.1f}")

# write to the .geo file
geo_content = f"""{sep}
xmin = {xmin};
xmax = {xmax};
ymin = {ymin};
ymax = {ymax};
zmin = {zmin};
zmax = {zmax};

"""

# Write to the file
with open(geofile, "w") as f:
    f.write(geo_content)

# Define the mesh sizes for different regions, in meters
dx = 200e3
dx_fault = 5e3
dx_fault_coarse = 20e3
dx_anomaly = 5e3
dx_anomaly_coarse = 20e3

geo_content = f"""{sep}
dx = {dx};
dx_fault = {dx_fault};
dx_fault_coarse = {dx_fault_coarse};
dx_anomaly = {dx_anomaly};
dx_anomaly_coarse = {dx_anomaly_coarse};

"""

# append to the file
with open(geofile, "a") as f:
    f.write(geo_content)

In [ ]:
# Same break points as original
break1_list = [
    9, 10, 293, 437, 581, 725, 869,
    1013, 1157, 1301, 1445, 1589, 1733, 1877, 2021, 2165, 2309, 2453, 2597, 2741, 2885, 3029, 3173,
    3317, 3461, 3605, 3749, 3893, 4037, 4181, 4325, 4469, 4613, 4757, 4901, 5045, 5189, 5333, 5477,
    5621, 5765, 5909, 6053, 6197, 6341, 6485, 6629, 6773, 6917, 7061, 7205, 7349, 7493, 7637, 7781,
    7925, 8069, 8213, 8357, 8501, 8645, 8789, 8933, 9077, 9221, 9365, 9509
]
break2_list = [
    177, 178, 377, 521, 665, 809, 953,
    1097, 1241, 1385, 1529, 1673, 1817, 1961, 2105, 2249, 2393, 2537, 2681, 2825, 2969, 3113, 3257,
    3401, 3545, 3689, 3833, 3977, 4121, 4265, 4409, 4553, 4697, 4841, 4985, 5129, 5273, 5417, 5561,
    5705, 5849, 5993, 6137, 6281, 6425, 6569, 6713, 6857, 7001, 7145, 7289, 7433, 7577, 7721, 7865,
    8009, 8153, 8297, 8441, 8585, 8729, 8873, 9017, 9161, 9305, 9449, 9593
]
end3_list = [
    287, 288, 432, 576, 720, 864, 1008,
    1152, 1296, 1440, 1584, 1728, 1872, 2016, 2160, 2304, 2448, 2592, 2736, 2880, 3024, 3168, 3312,
    3456, 3600, 3744, 3888, 4032, 4176, 4320, 4464, 4608, 4752, 4896, 5040, 5184, 5328, 5472, 5616,
    5760, 5904, 6048, 6192, 6336, 6480, 6624, 6768, 6912, 7056, 7200, 7344, 7488, 7632, 7776, 7920,
    8064, 8208, 8352, 8496, 8640, 8784, 8928, 9072, 9216, 9360, 9504, 9648
]
start1_list = [1, 4] + [x + 2 for x in end3_list[1:-1]]

print(f"Break lists lengths: {len(break1_list)}, {len(break2_list)}, {len(end3_list)}, {len(start1_list)}")

In [ ]:
def exo_fault_to_points_with_dx_rotated(points, break1_list, break2_list,
                                        geo_file=None,
                                        x_offset=0, y_offset=0,
                                        coarse_str="dx_fault_coarse",
                                        fine_str="dx_fault"):
    if geo_file is None:
        raise ValueError("geo_file must be specified")

    seg2_points = set()
    for idx, (b1, b2) in enumerate(zip(break1_list, break2_list), start=1):
        if idx in (1, 2):
            seg2_points.update(range(b1, b2 + 1, 2))
        else:
            seg2_points.update(range(b1, b2 + 1))

    with open(geo_file, "a") as f:
        for i, (x, y, z) in enumerate(points, start=1):
            dx_str = fine_str if i in seg2_points else coarse_str
            f.write(f"Point({i}) = {{{x-x_offset}, {y-y_offset}, {z}, {dx_str}}};\n")

    print(f"Wrote {geo_file} with {len(points)} points ({len(seg2_points)} points got '{fine_str}')")
    return len(points)

In [ ]:
npts = exo_fault_to_points_with_dx_rotated(
    points=points,
    break1_list=break1_list,
    break2_list=break2_list,
    geo_file=geofile,
    x_offset=x0,
    y_offset=y0,
    coarse_str="dx_fault_coarse",
    fine_str="dx_fault"
)

In [ ]:
# Create splines with the same logic as original
start_line_id = 1

for i, (b1, b2, e3) in enumerate(zip(break1_list, break2_list, end3_list)):
    line_id = start_line_id + i
    base_id = (line_id - 1) * 3 + 1

    if i == 0:
        seg1_expr = f"1, 2, 5:2:9"
        seg2_expr = f"9:2:177"
        seg3_expr = f"177:2:287"
    elif i == 1:
        seg1_expr = f"4, 3, 6:2:10"
        seg2_expr = f"10:2:178"
        seg3_expr = f"178:2:288"
    else:
        seg1_expr = f"{b1-3}, {b1-4}, {b1-2}:{b1}"
        seg2_expr = f"{b1}:{b2}"
        seg3_expr = f"{b2}:{e3}"

    ut.append_spline_to_geo(geofile, base_id, seg1_expr)
    ut.append_spline_to_geo(geofile, base_id + 1, seg2_expr)
    ut.append_spline_to_geo(geofile, base_id + 2, seg3_expr)

In [ ]:
# Update point and line count
start_point_id = npts + 1
start_line_id = (len(break1_list) * 3) + 1
print(f"Next line ID: {start_line_id}")

In [ ]:
# Process trench data with corrections
trench_file = datadir + "trench_xyz.txt"
trench_points = []

try:
    trench = pd.read_csv(trench_file, delim_whitespace=True, header=None, names=["x", "y"])
    
    # Apply the same rotation to trench data
    xy_trench = trench[['x', 'y']].values.T
    xy_trench_rotated = rotation_matrix @ xy_trench
    trench['x'] = xy_trench_rotated[0]
    trench['y'] = xy_trench_rotated[1]
    
    # Filter trench to plate horizontal range (x-direction)
    plate_x_min = points[:, 0].min()
    plate_x_max = points[:, 0].max()
    trench_filtered = trench[
        (trench['x'] >= plate_x_min * 0.95) & 
        (trench['x'] <= plate_x_max * 1.05)
    ].reset_index(drop=True)
    
    print(f"Trench points: {len(trench)} -> {len(trench_filtered)} (filtered to plate range)")
    
    if len(trench_filtered) > 0:
        # Sort by y-coordinate
        trench_filtered = trench_filtered.sort_values(by='y', ascending=False).reset_index(drop=True)
        
        # Write trench points at z=0 (surface)
        with open(geofile, "a") as f:
            f.write("\n// Trench points (at surface z=0)\n")
            for i, row in trench_filtered.iterrows():
                pid = start_point_id + i
                f.write(f"Point({pid}) = {{{row.x-x0:.6f}, {row.y-y0:.6f}, 0.0, dx_fault_coarse}};\n")
                trench_points.append(pid)
        
        start_point_id += len(trench_filtered)
        print(f"Added {len(trench_filtered)} trench points at z=0")
        
except FileNotFoundError:
    print(f"Trench file not found: {trench_file}")
except Exception as e:
    print(f"Error processing trench data: {e}")

In [ ]:
# Add boundary intersection points for complete domain enclosure
def add_boundary_intersections(points, start1_list, end3_list, x0, y0, 
                              xmin, xmax, ymin, ymax, zmin, zmax,
                              geo_file, start_point_id):
    boundary_points = {}
    
    with open(geo_file, "a") as f:
        f.write("\n// Boundary intersection points\n")
        
        # North boundary (ymax) intersections
        north_points = []
        for i, pid in enumerate(start1_list):
            x, y, z = points[pid-1]
            new_pid = start_point_id
            f.write(f"Point({new_pid}) = {{{x-x0}, {ymax}, {z}, dx}};\n")
            north_points.append(new_pid)
            start_point_id += 1
            
        # South boundary (ymin) intersections
        south_points = []
        for i, pid in enumerate(end3_list):
            x, y, z = points[pid-1]
            new_pid = start_point_id
            f.write(f"Point({new_pid}) = {{{x-x0}, {ymin}, {z}, dx}};\n")
            south_points.append(new_pid)
            start_point_id += 1
            
        # Bottom boundary (zmin) intersections
        bottom_points = []
        for i, pid in enumerate(end3_list):
            x, y, z = points[pid-1]
            new_pid = start_point_id
            f.write(f"Point({new_pid}) = {{{x-x0}, {y-y0}, {zmin}, dx}};\n")
            bottom_points.append(new_pid)
            start_point_id += 1
            
        # Domain corner points
        corners = [
            (xmin, ymax, zmax, "Top-NW"),
            (xmax, ymax, zmax, "Top-NE"), 
            (xmax, ymin, zmax, "Top-SE"),
            (xmin, ymin, zmax, "Top-SW"),
            (xmin, ymax, zmin, "Bottom-NW"),
            (xmax, ymax, zmin, "Bottom-NE"),
            (xmax, ymin, zmin, "Bottom-SE"),
            (xmin, ymin, zmin, "Bottom-SW"),
        ]
        
        corner_points = []
        f.write("\n// Domain corner points\n")
        for x, y, z, desc in corners:
            new_pid = start_point_id
            f.write(f"Point({new_pid}) = {{{x}, {y}, {z}, dx}};\n")
            corner_points.append(new_pid)
            start_point_id += 1
    
    boundary_points = {
        'north': north_points,
        'south': south_points,
        'bottom': bottom_points,
        'corners': corner_points
    }
    
    print(f"Added boundary points: North={len(north_points)}, South={len(south_points)}, Bottom={len(bottom_points)}, Corners={len(corner_points)}")
    
    return boundary_points, start_point_id

# Apply boundary intersections
boundary_points, start_point_id = add_boundary_intersections(
    points, start1_list, end3_list, x0, y0,
    xmin, xmax, ymin, ymax, zmin, zmax,
    geofile, start_point_id
)

In [ ]:
# Create vertical lines connecting fault segments
vertical_line_start_ids = []
vertical_line_break1_ids = []
vertical_line_break2_ids = []
vertical_line_end3_ids = []

with open(geofile, "a") as f:
    f.write("\n// Vertical connecting lines\n")
    
    # start1_list connections
    for i in range(len(start1_list) - 1):
        start_line_id += 1
        vertical_line_start_ids.append(start_line_id)
        f.write(f"Line({start_line_id}) = {{{start1_list[i]}, {start1_list[i+1]}}};\n")

    # break1_list connections
    for i in range(len(break1_list) - 1):
        start_line_id += 1
        vertical_line_break1_ids.append(start_line_id)
        f.write(f"Line({start_line_id}) = {{{break1_list[i]}, {break1_list[i+1]}}};\n")

    # break2_list connections
    for i in range(len(break2_list) - 1):
        start_line_id += 1
        vertical_line_break2_ids.append(start_line_id)
        f.write(f"Line({start_line_id}) = {{{break2_list[i]}, {break2_list[i+1]}}};\n")

    # end3_list connections
    for i in range(len(end3_list) - 1):
        start_line_id += 1
        vertical_line_end3_ids.append(start_line_id)
        f.write(f"Line({start_line_id}) = {{{end3_list[i]}, {end3_list[i+1]}}};\n")
        
    # Connect trench points if available
    trench_line_ids = []
    if len(trench_points) > 1:
        f.write("\n// Trench connecting lines\n")
        for i in range(len(trench_points) - 1):
            start_line_id += 1
            trench_line_ids.append(start_line_id)
            f.write(f"Line({start_line_id}) = {{{trench_points[i]}, {trench_points[i+1]}}};\n")
            
    # Connect shallowest plate points to trench (vertical lines to surface)
    plate_to_trench_lines = []
    if len(trench_points) > 0:
        f.write("\n// Lines from shallowest plate to trench\n")
        n_connections = min(len(start1_list), len(trench_points))
        for i in range(n_connections):
            start_line_id += 1
            plate_to_trench_lines.append(start_line_id)
            f.write(f"Line({start_line_id}) = {{{start1_list[i]}, {trench_points[i]}}};\n")

print(f"Added lines: vertical={len(vertical_line_start_ids)+len(vertical_line_break1_ids)+len(vertical_line_break2_ids)+len(vertical_line_end3_ids)}, trench={len(trench_line_ids)}, plate-to-trench={len(plate_to_trench_lines)}")

In [ ]:
# Build curve loops and surfaces
surface_id = 1
curve_loop_id = 1

with open(geofile, "a") as f:
    f.write("\n// Fault surface patches\n")
    for i in range(len(break1_list)-1):
        base_id = i * 3 + 1
        next_base_id = (i + 1) * 3 + 1

        # Segment 1 surface
        f.write(f"Curve Loop({curve_loop_id}) = {{{base_id}, {vertical_line_break1_ids[i]}, -{next_base_id}, -{vertical_line_start_ids[i]}}};\n")
        f.write(f"Surface({surface_id}) = {{{curve_loop_id}}};\n")
        curve_loop_id += 1
        surface_id += 1

        # Segment 2 surface
        f.write(f"Curve Loop({curve_loop_id}) = {{{base_id+1}, {vertical_line_break2_ids[i]}, -{next_base_id+1}, -{vertical_line_break1_ids[i]}}};\n")
        f.write(f"Surface({surface_id}) = {{{curve_loop_id}}};\n")
        curve_loop_id += 1
        surface_id += 1

        # Segment 3 surface
        f.write(f"Curve Loop({curve_loop_id}) = {{{base_id+2}, {vertical_line_end3_ids[i]}, -{next_base_id+2}, -{vertical_line_break2_ids[i]}}};\n")
        f.write(f"Surface({surface_id}) = {{{curve_loop_id}}};\n")
        curve_loop_id += 1
        surface_id += 1
        
    # Create surfaces connecting shallowest plate to trench
    if len(trench_points) > 1 and len(plate_to_trench_lines) > 1:
        f.write("\n// Surfaces from shallowest plate to trench\n")
        for i in range(len(plate_to_trench_lines) - 1):
            f.write(f"Curve Loop({curve_loop_id}) = {{{plate_to_trench_lines[i]}, {trench_line_ids[i]}, -{plate_to_trench_lines[i+1]}, -{vertical_line_start_ids[i]}}};\n")
            f.write(f"Surface({surface_id}) = {{{curve_loop_id}}};\n")
            curve_loop_id += 1
            surface_id += 1

print(f"Created {surface_id-1} surfaces")

In [ ]:
print(f"\n=== Corrected Geo file creation completed ===\n")
print(f"Output file: {geofile}")
print(f"Total fault points: {npts}")
print(f"Applied 45° counterclockwise rotation")
print(f"Fixed issues:")
print(f"  1. ✅ Trench points truncated to plate horizontal range")
print(f"  2. ✅ Trench z-coordinate fixed to z=0 (surface)")
print(f"  3. ✅ Trench points connected with lines and surfaces")
print(f"  4. ✅ Kept original domain bounds")
print(f"  5. ✅ Added boundary intersection points")
print(f"Total points: {start_point_id-1}")
print(f"Total surfaces: {surface_id-1}")